# Qwen2.5-Coder Fine-tuning on HTML/CSS Dataset
**Run each cell in order. Total time: ~2-3 hours on T4 GPU.**

Before starting:
1. Runtime → Change runtime type → **T4 GPU**
2. Have your HuggingFace token ready from https://huggingface.co/settings/tokens
3. No license approval needed — Qwen2.5-Coder is fully open

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install -q transformers trl peft accelerate bitsandbytes datasets huggingface_hub
print('✓ Dependencies installed')

In [ ]:
# ── Cell 2: Login to Hugging Face ─────────────────────────────────────────
import os
from huggingface_hub import login

HF_TOKEN    = 'hf_YOUR_TOKEN_HERE'  # ← paste your token from huggingface.co/settings/tokens
HF_USERNAME = 'your-username'       # ← your HuggingFace username
MODEL_NAME  = 'llama-html-codegen'  # ← name for your fine-tuned model repo

os.environ['HF_TOKEN'] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
print(f'✓ Logged in — model will be pushed to: {HF_USERNAME}/{MODEL_NAME}')

In [ ]:
# ── Cell 3: Upload your JSONL files ───────────────────────────────────────
from google.colab import files

print('Select all 5 files: part1.jsonl, part2.jsonl, part3.jsonl, part4.jsonl, part5.jsonl')
uploaded = files.upload()
print(f'✓ Uploaded: {list(uploaded.keys())}')

In [ ]:
# ── Cell 4: Load and prepare dataset ──────────────────────────────────────
# Qwen2.5 uses ChatML format: <|im_start|>role\ncontent<|im_end|>
import json
from datasets import Dataset

SYSTEM_MSG = (
    'You are a code generation assistant. Given a natural language description '
    'of a UI component, generate valid semantic HTML and CSS. Output ONLY the '
    'HTML wrapped in <html><head><style>...</style></head><body>...</body></html>. '
    'No explanation, no markdown fences.'
)

data = []
for i in range(1, 6):
    filename = f'part{i}.jsonl'
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                    if obj.get('prompt') and obj.get('response'):
                        # Qwen ChatML format
                        data.append({
                            'text': (
                                f'<|im_start|>system\n{SYSTEM_MSG}<|im_end|>\n'
                                f'<|im_start|>user\n{obj["prompt"]}<|im_end|>\n'
                                f'<|im_start|>assistant\n{obj["response"]}<|im_end|>'
                            )
                        })
                except json.JSONDecodeError:
                    continue
        print(f'✓ Loaded {filename}')
    except FileNotFoundError:
        print(f'✗ {filename} not found — skipping')

print(f'\nTotal samples: {len(data)}')
dataset = Dataset.from_list(data)
dataset = dataset.train_test_split(test_size=0.01, seed=42)
print(f'Train: {len(dataset["train"])} | Eval: {len(dataset["test"])}')

In [ ]:
# ── Cell 5: Load Qwen2.5-Coder-1.5B (no license needed, great at HTML/CSS) ─
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

BASE_MODEL = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading {BASE_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
print('✓ Model loaded')

In [ ]:
# ── Cell 6: Apply LoRA ─────────────────────────────────────────────────────
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expect: ~1-2% trainable — that's correct

In [ ]:
# ── Cell 7: Train ──────────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir='./qwen-html-checkpoints',
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=100,
    save_steps=1000,
    eval_strategy='steps',
    eval_steps=1000,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    max_seq_length=1024,
    report_to='none',
    save_total_limit=2,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    args=training_args,
)

print('Starting training... (~2-3 hours on T4)')
print('Do NOT close this tab!')
trainer.train()
print('✓ Training complete!')

In [ ]:
# ── Cell 8: Push to Hugging Face Hub ──────────────────────────────────────
REPO_ID = f'{HF_USERNAME}/{MODEL_NAME}'

print(f'Pushing to {REPO_ID}...')
model.push_to_hub(REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(REPO_ID, token=HF_TOKEN)
print(f'✓ Model pushed to: https://huggingface.co/{REPO_ID}')
print()
print('=' * 50)
print('Add these to your generative-code-playground/.env:')
print('=' * 50)
print(f'VITE_HF_API_KEY={HF_TOKEN}')
print(f'VITE_HF_MODEL={REPO_ID}')

In [ ]:
# ── Cell 9: Quick test ─────────────────────────────────────────────────────
from transformers import pipeline

pipe = pipeline('text-generation', model=model, tokenizer=tokenizer, max_new_tokens=512)

test_prompt = (
    '<|im_start|>system\n'
    'You are a code generation assistant. Output only HTML/CSS.<|im_end|>\n'
    '<|im_start|>user\n'
    'Create a blue button with rounded corners<|im_end|>\n'
    '<|im_start|>assistant\n'
)

result = pipe(test_prompt, do_sample=True, temperature=0.7)
generated = result[0]['generated_text'].split('<|im_start|>assistant\n')[-1]
print('Generated output:')
print(generated)